**Goal**: This notebook parses PDB filenames from `reps_all_pdbs/` to build a dataframe of representatives, then organises the files into per-MC subfolders.

In [1]:
import os
import shutil
import pandas as pd

# I. Parse PDB filenames into a DataFrame

In [2]:
pdb_dir = "/u/mdmc/enyanduk/internship_areasciencepark/Data/dpcstruct/reps_all_pdbs"

records = []
for filename in os.listdir(pdb_dir):
    if not filename.endswith(".pdb"):
        continue
    # e.g. filename = "A0A009K0I0_6579.pdb"
    representative = filename  # A0A009K0I0_6579.pdb
    name_no_ext = filename[:-4]  # A0A009K0I0_6579
    last_underscore = name_no_ext.rfind("_")
    protein = name_no_ext[:last_underscore]  # A0A009K0I0
    mc_num = name_no_ext[last_underscore + 1:]  # 6579
    mcid = mc_num  # 6579
    records.append({"representative": representative, "protein": protein, "mcid": mcid})

df_pdbs = pd.DataFrame(records)
print(f"Shape: {df_pdbs.shape}")
df_pdbs.head()

Shape: (56438, 3)


,representative,protein,mcid
0,A0A3S3Q9K1_46428.pdb,A0A3S3Q9K1,46428
1,A0A1Y1IIT2_32419.pdb,A0A1Y1IIT2,32419
2,A0A453JID8_13567.pdb,A0A453JID8,13567
3,A0A453JFQ9_18197.pdb,A0A453JFQ9,18197
4,A0A7R8L786_5907.pdb,A0A7R8L786,5907


# II. Sort by MCID

In [3]:
# Some transformations to the dataframe:
df = df_pdbs.copy()
# mcid is currently a string, we convert it to int for sorting
df["mcid"] = df["mcid"].astype(int)
# T1 : We sort the data by mcid in ascending order
df.sort_values(by="mcid",ascending=True,inplace=True)
# T2 : We reset the index
df.reset_index(drop=True, inplace=True)
# T3 : Rewrite each ID in mcid column as MCID: e.g.: 1 -> MC1
df["mcid"] = df["mcid"].apply(lambda x: f"MC{x}")
# We organize the columns in the order: mcid, protein, representative
df = df[["mcid", "protein", "representative"]]
# Head of the transformed dataframe
df.head(10)

,mcid,protein,representative
0,MC0,A0A537IMV5,A0A537IMV5_0.pdb
1,MC0,A0A1Q3ZAL7,A0A1Q3ZAL7_0.pdb
2,MC1,A0A7Y5H9I3,A0A7Y5H9I3_1.pdb
3,MC1,A0A497KZ75,A0A497KZ75_1.pdb
4,MC2,A0A7Y5HAR7,A0A7Y5HAR7_2.pdb
5,MC2,A0A2A2QTX3,A0A2A2QTX3_2.pdb
6,MC3,A0A849WCM9,A0A849WCM9_3.pdb
7,MC3,A0A2E0QQW9,A0A2E0QQW9_3.pdb
8,MC4,A0A849WD35,A0A849WD35_4.pdb
9,MC4,A0A372JJE4,A0A372JJE4_4.pdb


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56438 entries, 0 to 56437
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   mcid            56438 non-null  object
 1   protein         56438 non-null  object
 2   representative  56438 non-null  object
dtypes: object(3)
memory usage: 1.3+ MB


# III. Save the DataFrame

In [5]:
save_path = "/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_reps_pdbs.csv"
df.to_csv(save_path, index=False)
print(f"Saved df ({df.shape}) to {save_path}")

Saved df ((56438, 3)) to /u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_reps_pdbs.csv


# IV. Create per-MC subfolders with PDB files
For each unique `mcid`, create a subfolder `MCID_pdb/` and copy the original `.pdb` files into it.

In [6]:
output_base = "/u/mdmc/enyanduk/internship_areasciencepark/Data/dpcstruct/dpcstruct_reps_pdbs"
os.makedirs(output_base, exist_ok=True)

for mcid, group in df.groupby("mcid"):
    mc_folder = os.path.join(output_base, f"{mcid}_pdb")
    os.makedirs(mc_folder, exist_ok=True)
    for _, row in group.iterrows():
        src = os.path.join(pdb_dir, row["representative"])
        dst = os.path.join(mc_folder, row["representative"])
        shutil.copy2(src, dst)

n_folders = df["mcid"].nunique()
print(f"Created {n_folders} subfolders in {output_base}")

Created 28246 subfolders in /u/mdmc/enyanduk/internship_areasciencepark/Data/dpcstruct/dpcstruct_reps_pdbs
